# 프롬프트 엔지니어링

## [내용]
- 출력의 예시를 제공하는 형태 : zero-shot, one-shot, few-shot
- 사고의 단계 (Chain-of-Thought)
    - 개념: 모델에게 "단계별로 생각해보자"고 요청하여 논리적 비약을 막고 정확도를 높이는 기법
- 신뢰의 기술 (Self-Consistency)
    - 개념: 동일한 질문을 여러 번 던져 가장 많이 나오는 결론을 선택함으로써 답변의 일관성을 확보함
- 추론과 행동 (ReAct 기초)
    - 개념: AI가 스스로 생각(Thought)하고, 행동(Action)을 결정하는 자율적 흐름을 구축함.
- 로직의 흐름 (Prompt Chaining)
    - 개념: 복잡한 업무를 작은 단위의 프롬프트로 쪼개어 순차적으로 연결함.

In [1]:
# 설치 라이브러리
# !pip install -U langchain langchain-openai langchain-google-genai

# 환경변수 로딩

In [23]:
# api key 설정
from dotenv import load_dotenv
import os

load_dotenv(override=True)
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

# 출력의 예시를 제공하는 형태
- zero-shot, one-shot, few-shot

# 사고의 단계
- 개념: 모델에게 "단계별로 생각해보자"고 요청하여 논리적 비약을 막고 정확도를 높이는 기법

In [33]:
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# llm = ChatOpenAI(model="gpt-4o")
llm = init_chat_model(
    api_key=GEMINI_API_KEY,
    model="models/gemini-2.5-flash",   
    model_provider="google-genai",
    temperature=0.5 
    )

# [실습] Zero-shot CoT: 마법의 구문 "Step-by-step"
cot_prompt = ChatPromptTemplate.from_template(
    "우리 회사의 지난분기 이탈 고객이 15% 증가했습니다. 원인을 분석하고 대응 전략을 '단계별로' 세워주세요."
)

chain = cot_prompt | llm | StrOutputParser()

result = chain.invoke({})

In [34]:
from IPython.display import display, Markdown
print("--- [Zero-shot CoT 결과] ---")
# 마크다운으로 렌더링하여 출력
display(Markdown(result))

--- [Zero-shot CoT 결과] ---


우리 회사의 지난 분기 이탈 고객 15% 증가는 매우 심각한 신호이며, 신속하고 체계적인 분석 및 대응이 필요합니다. 아래에 원인 분석 방안과 단계별 대응 전략을 제시합니다.

---

### **1. 원인 분석 (Diagnostic Phase)**

이탈률 15% 증가는 특정 이벤트나 문제에서 비롯되었을 가능성이 높습니다. 정확한 원인 파악을 위해 다음 데이터를 우선적으로 분석해야 합니다.

**가. 데이터 기반 분석:**

1.  **이탈 고객 특성 분석:**
    *   **세그먼트:** 어떤 고객층에서 이탈이 급증했는가? (신규/기존 고객, 고가치/저가치 고객, 특정 상품/서비스 이용 고객, 지역, 산업군 등)
    *   **가입 시점:** 최근 가입한 고객들의 이탈이 많은가, 아니면 오래된 고객들의 이탈이 많은가?
    *   **사용 패턴:** 이탈 직전 고객들의 서비스/제품 사용량 변화 (접속 빈도, 사용 기능, 구매 주기 등)
    *   **이탈 직전 행동:** 특정 기능 사용 중단, 고객 지원 문의 증가, 결제 실패 등.

2.  **고객 피드백 분석:**
    *   **이탈 설문조사/인터뷰:** 이탈 시 고객이 직접 남긴 피드백(원인) 분석 (만약 없다면 즉시 도입 필요).
    *   **고객 지원 기록:** 이탈 고객들의 과거 문의 내역, 불만 사항, 해결 여부 및 시간.
    *   **VOC (Voice of Customer) 채널:** 웹사이트 피드백, 소셜 미디어, 앱 리뷰 등에서 발견되는 공통적인 불만사항.

3.  **내부 요인 분석:**
    *   **제품/서비스 변경:** 지난 분기에 있었던 주요 제품 업데이트, 버그 발생, UI/UX 변경, 기능 추가/삭제 등. (특히 부정적 영향을 미친 변경사항이 있었는지 확인)
    *   **가격/프로모션 정책 변경:** 구독료 인상, 할인 정책 변경, 혜택 축소 등.
    *   **고객 서비스 품질 변화:** 상담 대기 시간 증가, 문제 해결 능력 저하, 상담원 불친절 등.
    *   **영업/온보딩 프로세스 변화:** 신규 고객 유치 과정에서 잘못된 기대치를 심어준 부분은 없는가?

4.  **외부 요인 분석:**
    *   **경쟁사 동향:** 경쟁사의 신규 서비스 출시, 파격적인 프로모션, 가격 정책 변화 등.
    *   **시장 상황 변화:** 경기 침체, 산업 트렌드 변화, 법규 변경 등으로 인한 고객 니즈 변화.

**나. 가설 수립 및 검증:**

위 데이터 분석을 통해 이탈의 주요 원인에 대한 가설을 세우고, 추가적인 데이터를 통해 이를 검증합니다. (예: "지난달 제품 업데이트 후 특정 기능 버그로 인한 이탈 증가" -> 버그 신고 내역과 이탈 고객들의 해당 기능 사용 여부 확인)

---

### **2. 단계별 대응 전략 (Step-by-Step Response Strategy)**

원인 분석을 바탕으로 단기, 중기, 장기적인 관점에서 단계별 대응 전략을 수립합니다.

#### **1단계: 긴급 대응 및 분석 심화 (1~2주)**

*   **목표:** 이탈 증가세 늦추기 및 원인 정확히 파악 완료.
*   **세부 전략:**
    1.  **Cross-Functional 태스크포스 (TFT) 구성:** 제품, 마케팅, 고객 서비스, 영업 등 유관 부서 핵심 인력으로 TFT를 구성하여 신속한 의사결정 및 실행을 추진합니다.
    2.  **데이터 대시보드 구축:** 이탈 관련 핵심 지표(Churn Rate, Retention Rate, Usage Data, CSAT 등)를 실시간으로 모니터링할 수 있는 대시보드를 구축합니다.
    3.  **이탈 고객 대상 긴급 설문조사/인터뷰 강화:** 이탈 의사를 밝힌 고객 또는 최근 이탈한 고객들에게 직접 연락하여 이탈 사유를 상세히 듣습니다. (소액의 보상 제공도 고려)
    4.  **확인된 치명적 문제 즉시 해결:** 분석 결과 명확하게 드러난 치명적인 버그나 고객 서비스 문제를 최우선적으로 해결합니다. (예: 특정 결제 오류, 서비스 접속 불량 등)
    5.  **이탈 가능성 높은 고객군 식별 및 사전 접촉:** 데이터를 기반으로 현재 이탈 위험이 높은 고객(예: 사용량이 급감한 고객, 불만 문의를 자주 한 고객)을 식별하여 선제적으로 연락하고 문제 해결을 돕거나 특별 혜택을 제공합니다.

#### **2단계: 단기적 이탈 방지 및 고객 만족도 개선 (1~3개월)**

*   **목표:** 이탈률을 이전 수준으로 회복시키고 고객 만족도 회복.
*   **세부 전략:**
    1.  **핵심 이탈 원인에 대한 집중 개선:**
        *   **제품/서비스 문제:** 분석 결과 가장 큰 문제로 지목된 기능 개선, 버그 수정, UX 개선 등을 최우선으로 진행하고 고객들에게 적극적으로 개선 사실을 알립니다.
        *   **고객 서비스 문제:** 상담 인력 충원, 교육 강화, 챗봇/FAQ 시스템 개선, 응대 매뉴얼 보완 등을 통해 고객 경험을 향상시킵니다.
        *   **가격/가치 문제:** 경쟁사 분석을 바탕으로 가격 경쟁력을 재검토하고, 고객이 느끼는 서비스의 '가치'를 높이기 위한 추가 혜택 제공 또는 커뮤니케이션 강화 (서비스의 장점, 활용 팁 등)
    2.  **재활성화 캠페인:** 이탈 고객 중 일부를 대상으로 특별 할인, 무료 체험 기간 연장, 개인화된 사용 가이드 제공 등을 통해 재유입을 유도합니다.
    3.  **고객 성공(Customer Success) 강화:** 고객의 서비스 활용을 돕는 온보딩 프로세스를 강화하고, 정기적인 고객 건강 점수(Customer Health Score) 관리를 통해 문제가 발생하기 전에 개입합니다.
    4.  **커뮤니케이션 채널 활성화:** 고객들이 언제든 쉽게 불만을 제기하고 피드백을 줄 수 있는 창구를 확대하고, 피드백에 대한 빠른 응대와 해결을 보여줍니다.

#### **3단계: 중장기적 고객 유지 및 성장 전략 구축 (3개월 이상)**

*   **목표:** 이탈률을 지속적으로 관리하고, 고객 생애 가치(CLTV)를 극대화하며, 고객을 옹호자로 전환.
*   **세부 전략:**
    1.  **데이터 기반 예측 모델 구축:** 머신러닝 등을 활용하여 이탈 위험이 높은 고객을 사전에 예측하고, 예측 결과에 따라 맞춤형 개입 전략을 자동화합니다.
    2.  **지속적인 제품/서비스 혁신:** 고객 피드백과 시장 트렌드를 반영하여 경쟁 우위를 유지할 수 있는 차별화된 기능 및 서비스를 꾸준히 개발합니다.
    3.  **강력한 고객 커뮤니티 구축:** 고객들이 서로 정보를 공유하고, 회사와 소통할 수 있는 커뮤니티를 활성화하여 소속감과 충성도를 높입니다.
    4.  **개인화된 고객 경험 제공:** 고객의 사용 패턴, 선호도에 기반한 개인화된 추천, 맞춤형 프로모션, 콘텐츠 제공 등을 통해 고객 만족도를 극대화합니다.
    5.  **온보딩 프로세스 최적화:** 신규 고객이 서비스의 가치를 빠르게 인지하고 활용할 수 있도록 온보딩 과정을 지속적으로 개선하고 개인화합니다.
    6.  **직원 교육 및 문화 강화:** 모든 직원이 고객 중심 마인드를 가질 수 있도록 교육을 강화하고, 고객 만족을 최우선 가치로 여기는 기업 문화를 정착시킵니다.
    7.  **지속적인 성과 측정 및 개선:** 설정된 KPI(핵심 성과 지표)를 지속적으로 측정하고, 각 단계별 전략의 효과를 분석하여 다음 개선 주기에 반영하는 반복적인 프로세스를 구축합니다.

---

**결론적으로,** 지난 분기 이탈 고객 증가는 위기이자 성장의 기회가 될 수 있습니다. 신속하고 정확한 원인 분석을 통해 근본적인 문제를 해결하고, 단계적인 전략 실행을 통해 고객 이탈을 방지하며 장기적으로는 고객 충성도를 높이는 계기로 삼아야 할 것입니다. 모든 과정에서 **데이터 기반의 의사결정**과 **부서 간 긴밀한 협력**이 필수적입니다.

# 신뢰의 기술 (Self-Consistency)
- 개념: 동일한 질문을 여러 번 던져 가장 많이 나오는 결론을 선택함으로써 답변의 일관성을 확보

In [ ]:
from langchain_core.runnables import RunnableParallel

# 온도를 높여 다양한 사고 경로 유도
llm_creative = ChatOpenAI(model="gpt-4o", temperature=0.7)

consistency_prompt = ChatPromptTemplate.from_template(
    "A사에서 생성형 AI를 도입할 때, 자체 서버(On-premise)와 클라우드 중 어느 것이 장기적으로 유리할까요?"
)

# 3개의 독립적인 추론 경로를 병렬로 실행
consistency_chain = RunnableParallel(
    path1=consistency_prompt | llm_creative | StrOutputParser(),
    path2=consistency_prompt | llm_creative | StrOutputParser(),
    path3=consistency_prompt | llm_creative | StrOutputParser()
)

print("--- [Self-Consistency 실행] ---")
result = consistency_chain.invoke({})

In [ ]:
print(f"결과 1: {result['path1'][:100]}...")
print(f"결과 2: {result['path2'][:100]}...")
print(f"결과 3: {result['path3'][:100]}...")

--- [Self-Consistency 실행] ---
결과 1: A사에서 생성형 AI를 도입할 때, 자체 서버(On-premise)와 클라우드 중 어느 것이 장기적으로 유리한지는 여러 요인에 따라 다릅니다. 각각의 장점과 단점을 고려하여 결정하...
결과 2: 생성형 AI를 도입할 때 자체 서버(On-premise)와 클라우드 중 어떤 것이 더 유리한지는 여러 요인에 따라 달라질 수 있습니다. 다음은 각각의 장단점을 고려하여 장기적인 관...
결과 3: A사에서 생성형 AI를 도입할 때 자체 서버(On-premise)와 클라우드 중 어느 것이 장기적으로 유리한지는 여러 가지 요인에 따라 달라질 수 있습니다. 각 옵션의 장단점을 고...


In [5]:
print(result['path1'])

생성형 AI를 도입할 때 자체 서버(On-premise)와 클라우드 중 어느 쪽이 장기적으로 유리한지는 여러 요인에 따라 달라질 수 있습니다. 다음은 각 옵션의 장단점을 고려할 때 유의해야 할 몇 가지 주요 사항입니다.

### 자체 서버(On-premise)

#### 장점:
1. **데이터 보안 및 프라이버시**: 민감한 데이터를 자체 서버에 보관함으로써 보안과 프라이버시를 강화할 수 있습니다.
2. **맞춤형 환경**: 기업의 특정 요구사항에 맞춰 서버 환경을 커스터마이즈할 수 있습니다.
3. **비용 예측 가능성**: 초기 구축 비용은 높지만, 장기적으로 운영 비용이 예측 가능할 수 있습니다.

#### 단점:
1. **초기 비용**: 하드웨어 및 인프라 구축에 대한 높은 초기 비용이 발생합니다.
2. **유지보수**: 서버 유지보수 및 업그레이드에 대한 책임이 기업에 있습니다.
3. **확장성의 한계**: 필요에 따라 쉽게 확장하기 어렵고, 추가적인 하드웨어 투자가 필요합니다.

### 클라우드

#### 장점:
1. **확장성**: 필요에 따라 자원을 쉽게 확장하거나 축소할 수 있어 유연성이 높습니다.
2. **초기 비용 절감**: 초기 하드웨어 투자 없이 월별 사용료로 비용을 관리할 수 있습니다.
3. **최신 기술 접근성**: 클라우드 제공업체들은 최신 기술과 업데이트를 제공하므로, 이에 대한 접근이 용이합니다.

#### 단점:
1. **데이터 보안 우려**: 데이터가 외부 서버에 저장되므로 보안 및 프라이버시 문제에 대한 우려가 있을 수 있습니다.
2. **지속적인 비용**: 사용량에 따라 지속적인 비용이 발생하며, 장기적으로는 비용이 증가할 수 있습니다.
3. **의존성**: 특정 클라우드 공급업체에 대한 의존성이 생길 수 있습니다.

### 결론

- **보안 및 프라이버시**가 최우선인 경우: 자체 서버가 더 적합할 수 있습니다.
- **유연성 및 최신 기술**에 대한 접근이 중요한 경우: 클라우드가 더 유리할 수 있습니다.
- **비용

In [6]:
print(result['path2'])

A사에서 생성형 AI를 도입할 때, 자체 서버(On-premise)와 클라우드 중 어느 것이 장기적으로 유리한지 결정하는 것은 여러 요소에 따라 달라집니다. 다음은 두 가지 옵션의 장단점을 고려하여 장기적으로 유리한 선택을 하는 데 도움이 될 수 있는 요소들입니다.

### 자체 서버(On-premise)

#### 장점
1. **데이터 보안 및 프라이버시**: 민감한 데이터를 외부 서버에 저장하지 않으므로 데이터 유출 위험이 적습니다.
2. **맞춤형 설정**: 하드웨어와 소프트웨어를 기업의 특정 요구에 맞게 완전히 맞춤화할 수 있습니다.
3. **제어권**: 시스템에 대한 완전한 제어권을 가질 수 있어, 필요에 따라 최적화 및 조정이 가능합니다.

#### 단점
1. **높은 초기 비용**: 하드웨어 구매 및 설치 비용이 많이 들며, 유지보수 비용도 발생합니다.
2. **확장성 제한**: 필요에 따라 빠르게 자원을 확장하는 것이 어려울 수 있습니다.
3. **운영 및 유지보수 부담**: IT 팀이 시스템의 운영 및 유지보수를 직접 관리해야 합니다.

### 클라우드

#### 장점
1. **비용 효율성**: 초기 투자 비용이 낮고, 사용한 만큼만 비용을 지불하는 구조입니다.
2. **확장성**: 필요에 따라 쉽게 자원을 확장하거나 축소할 수 있습니다.
3. **빠른 배포**: 새로운 기능이나 업데이트를 빠르게 적용할 수 있습니다.

#### 단점
1. **데이터 보안 우려**: 외부 서버에 데이터를 저장하므로 보안 및 프라이버시 문제가 발생할 수 있습니다.
2. **종속성**: 클라우드 서비스 제공업체에 대한 종속성이 발생할 수 있습니다.
3. **비용 변동성**: 사용량이 늘어남에 따라 비용이 예측하기 어려울 수 있습니다.

### 결론

- **데이터 보안 및 규제 준수가 중요한 경우**: 자체 서버가 유리할 수 있습니다.
- **비용 절감, 유연성 및 신속한 확장이 중요한 경우**: 클라우드가 더 적합할 수 있습니다.
- **혼합 모델**: 일부 민감

# 추론과 행동 (ReAct 기초)
- 개념: AI가 스스로 생각(Thought)하고, 행동(Action)을 결정하는 자율적 흐름을 구축함.

In [ ]:
react_system = """너는 전략적 AI 에이전트다. 질문에 대해 다음 형식을 엄격히 지켜라.
1. Thought: 현재 질문에 대해 분석한다.
2. Action: '직접 답변' 또는 '외부 검색 필요' 중 하나를 선택한다.
3. Response: 최종 답변을 작성한다.

질문: {input}"""

react_prompt = ChatPromptTemplate.from_template(react_system)
react_chain = react_prompt | llm | StrOutputParser()

--- [ReAct 사고 과정 확인] ---
1. Thought: 현재 NVIDIA의 주가 수익비율(PER)을 알아야 투자 가치에 대한 분석이 가능하다. 그러나 실시간 금융 데이터를 제공할 수 있는 능력이 없다.
2. Action: 외부 검색 필요
3. Response: NVIDIA의 최신 주가 수익비율(PER)은 실시간 금융 데이터를 제공하는 웹사이트나 애플리케이션을 통해 확인하실 수 있습니다. 확인 후, 일반적으로 PER이 낮을수록 주가가 상대적으로 저평가돼 있을 가능성이 있다고 판단되지만, 이는 업계 평균, 회사의 성장 가능성 등 다양한 요소를 고려해 판단해야 합니다.


In [12]:
result = react_chain.invoke({"input": "현재 엔비디아(NVIDIA)의 주가 수익비율(PER)을 바탕으로 투자 가치를 분석해줘."})

In [13]:
print("--- [ReAct 사고 과정 확인] ---")
print(result)

--- [ReAct 사고 과정 확인] ---
1. Thought: 주가 수익비율(PER)은 주가를 회사의 주당 순이익(EPS)으로 나눈 값으로, 주식이 저평가되어 있는지 판단하는데 사용된다. 최신 PER 값을 알기 위해서는 현재 엔비디아(NVIDIA)의 주가와 그에 따른 EPS에 대한 데이터가 필요하다. 이런 정보는 주로 실시간 금융 데이터 제공 업체에서 얻을 수 있다.

2. Action: 외부 검색 필요

3. Response: 현재 엔비디아(NVIDIA)의 주가 수익비율(PER)을 분석하려면 최신 주가와 EPS에 대한 정보가 필요합니다. 이러한 데이터를 확인하기 위해서는 금융 뉴스나 증권 관련 웹사이트를 통해 최신 정보를 참조하시길 권장합니다. PER은 상대적인 가치 평가 도구로, 업계 평균 PER과 비교하거나 회사의 역사적 PER와 비교하여 투자 가치를 판단할 수 있습니다.


# 비즈니스 로직 통합 설계
- DX 컨설턴트가 되어 '전통 시장 디지털 전환 솔루션'을 설계하는 통합 체인을 만들기

In [14]:
# 1. 현황 분석 체인
analysis_step = ChatPromptTemplate.from_template("{market_name}의 디지털화 저해 요인을 분석해줘.")

# 2. 솔루션 도출 체인
solution_step = ChatPromptTemplate.from_template("다음 분석을 바탕으로 소상공인을 위한 AX(AI 전환) 로드맵을 작성해줘: {analysis}")

# 통합 비즈니스 로직
ax_consulting_chain = (
    {"analysis": analysis_step | llm | StrOutputParser()}
    | solution_step
    | llm
    | StrOutputParser()
)

In [15]:
print("--- [DX 컨설팅 통합 리포트] ---")
print(ax_consulting_chain.invoke({"market_name": "남대문 시장"}))

--- [DX 컨설팅 통합 리포트] ---
남대문시장의 디지털 전환을 지원하기 위한 소상공인 AX(AI 전환) 로드맵을 다음과 같이 제안합니다:

### 1. 기초 디지털 역량 강화
- **디지털 기술 교육 및 워크숍**: 
  - 상인들을 대상으로 한 기초적인 디지털 기술 교육 프로그램을 마련합니다.
  - 소셜 미디어 활용법, 온라인 마케팅, 디지털 결제 시스템 사용법 등을 포함한 워크숍 제공.
- **멘토링 프로그램**:
  - 디지털 전환 경험이 있는 비즈니스 전문가들과의 멘토링 매칭 프로그램 운영.

### 2. 기존 거래 방식의 개선
- **병행 운영 전략**:
  - 전통적 거래 방식과 디지털 거래 방식을 병행할 수 있는 전략 고안.
  - 현금 결제와 디지털 결제가 모두 가능한 환경 조성.

### 3. 인프라 개발
- **기술 인프라 확충**:
  - 시장 전역에 안정적인 인터넷 연결 및 Wi-Fi 시설 구축.
  - 소상공인 맞춤형 온라인 플랫폼 개발 비용 지원.
- **디지털 결제 시스템 도입 지원**:
  - 소상공인들이 부담 없이 디지털 결제를 시작할 수 있도록 시스템 도입비 지원.

### 4. 경제적 지원 제도 마련
- **초기 투자 비용 지원**:
  - 디지털 전환을 위한 초기 투자비를 보조하거나, 저리의 대출 프로그램 제공.
- **실적 기반 보조금**:
  - 디지털화 실적에 기반한 보조금 제도 도입.

### 5. 정책 및 제도 강화
- **정부 및 지방자치단체의 지원 확대**:
  - 디지털 전환 촉진을 위한 세제 혜택 및 지원 정책 제시.
  - 상인을 위한 전담 지원팀 운영.
- **협력 네트워크 구축**:
  - 상인, 지방자치단체, 기술 제공자 간의 협력 체계 구축.

### 6. 소비자 인식 전환
- **소비자 교육 및 홍보**:
  - 디지털 결제의 장점과 편리함을 강조하는 캠페인 전개.
- **소비자 참여 이벤트**:
  - 전자 결제 시 할인 또는 리워드 제공 등을 통한 참여 유도.

### 7. 경쟁력 강화

# 모델별 추론 비교(GPT vs Gemini)

In [27]:
from langchain.chat_models import init_chat_model

gpt = init_chat_model(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini",
    model_provider="openai",
    temperature=0.5
    )
gemini = init_chat_model(
    api_key=GEMINI_API_KEY,
    model="models/gemini-2.5-flash",
    model_provider="google-genai",
    temperature=0.5
    )

def run_comparison(topic, model):
    market_name = {"market_name": topic}
    # GPT-4o와 Gemini 1.5를 사용한 결과 비교
    model_chain = {"analysis": analysis_step | model | StrOutputParser()} | solution_step | model | StrOutputParser()
    model_result = model_chain.invoke(market_name)

    return model_result

In [28]:
# gpt 모델 추론
gpt_result = run_comparison("오프라인 중고 서점", gpt)

In [29]:
# gemini 모델 추론
gemini_result = run_comparison("오프라인 중고 서점", gemini)

In [31]:
# 결과 출력
print(f"## GPT-4o 결과:\n{gpt_result}\n")
print(f"## Gemini 1.5-flash 결과:\n{gemini_result}")

## GPT-4o 결과:
소상공인을 위한 AX(AI 전환) 로드맵은 다음과 같은 단계로 구성될 수 있습니다. 이 로드맵은 중고 서점의 디지털화 저해 요인을 극복하고, AI 기술을 활용하여 비즈니스 모델을 혁신하는 데 초점을 맞추고 있습니다.

### 1. 현황 분석 및 목표 설정
- **기술적 인프라 분석**: 현재 보유하고 있는 기술 인프라를 평가하고, 필요한 기술적 요소를 파악합니다.
- **비즈니스 모델 재정의**: 전통적인 비즈니스 모델을 분석하고, 디지털화에 적합한 새로운 비즈니스 모델을 설정합니다.
- **목표 설정**: 디지털화의 목표(예: 온라인 판매 비율 증가, 고객층 확대 등)를 명확히 합니다.

### 2. 기술 교육 및 인프라 구축
- **기술 교육 프로그램 운영**: 직원 및 소상공인 대상으로 디지털 기술 및 AI에 대한 교육을 실시합니다.
- **기술 파트너십 구축**: IT 기업이나 스타트업과 협력하여 필요한 기술적 인프라를 구축합니다. 예를 들어, 웹사이트 제작, 온라인 쇼핑몰 구축, 재고 관리 시스템 도입 등을 지원받습니다.

### 3. 자본 지원 및 투자 유치
- **정부 및 민간 지원 프로그램 활용**: 소상공인을 위한 디지털 전환 지원금, 보조금, 대출 프로그램 등을 활용하여 초기 투자 비용을 줄입니다.
- **크라우드 펀딩 또는 투자 유치**: 디지털화에 대한 사업 계획서를 작성하고, 외부 투자자를 유치합니다.

### 4. 고객층 변화 및 마케팅 전략 개발
- **고객 데이터 분석**: 기존 고객 데이터를 분석하여 고객의 디지털 사용 패턴을 이해합니다.
- **디지털 마케팅 전략 수립**: SNS, 이메일 마케팅, 검색 엔진 최적화(SEO) 등을 활용하여 새로운 고객층을 타겟팅합니다.
- **고객 참여 유도**: 온라인 이벤트, 프로모션 등을 통해 고객의 관심을 유도하고, 디지털 서비스에 대한 수용성을 높입니다.

### 5. 재고 관리 시스템 디지털화
- **재고 관리 시스템 도입**: 다양한 책과 상품을 효율적으로 